# HR 域数据合成 Notebook

基于 BASE 域生成的组织架构数据，为 HR 域生成：
- Data Schema（JSON Schema）
- Data Models（Pydantic 模型）
- Database Content（Faker 脚本生成，固定 seed 可复现）
- Tools（工具函数类）
- Tasks（含难度分级、精确 nl_assertions，改进版）

### 初始化

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import json as _json
import subprocess
import sys
import time
from pathlib import Path
from synthesis.llm_client import LLMClient
from synthesis.data_utils import (
    save_generated_file, load_json, extract_code_block,
    verify_python_syntax, verify_json_structure,
    parse_ref_fields, topological_sort, validate_foreign_keys,
    calculate_coverage_statistics, format_validation_report,
    sample_db,
)

DOMAIN = "HR"
llm_model_name = "claude-opus-4-6"
llm = LLMClient(llm_model_name)

# 路径配置：生成代码和数据分离存放
PROMPTS_DIR       = Path("prompts") / DOMAIN
GENERATED_DATA_DIR = Path("generated") / "data_dir/domains" / DOMAIN
GENERATED_CODE_DIR = Path("generated") / "domains" / DOMAIN
BASE_DATA_DIR      = Path("generated") / "data_dir/domains" / "BASE"
BASE_CODE_DIR      = Path("generated") / "domains" / "BASE"
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_CODE_DIR.mkdir(parents=True, exist_ok=True)

print("✓ 初始化完成")
print(f"LLM: {llm.llm.model_name}")
print(f"Prompts 目录: {PROMPTS_DIR}")
print(f"生成数据目录：{GENERATED_DATA_DIR}")
print(f"生成代码目录：{GENERATED_CODE_DIR}")
print(f"BASE 数据目录：{BASE_DATA_DIR}")

✓ 初始化完成
LLM: claude-opus-4-6
Prompts 目录: prompts/HR
生成数据目录：generated/data_dir/domains/HR
生成代码目录：generated/domains/HR
BASE 数据目录：generated/data_dir/domains/BASE


### 0. 加载 BASE 上下文

In [ ]:
# 加载 BASE 域生成结果（作为 HR 域的基础上下文）
with open(BASE_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    base_schema = f.read()
with open(BASE_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    base_models = f.read()
with open(BASE_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_data_raw = f.read()

with open(BASE_CODE_DIR / "gen_data.py", 'r', encoding='utf-8') as f:
    base_gen_data_code = f.read()

base_data = json.loads(base_data_raw)
print("加载完成")

加载完成


### 1. 生成 HR Schema

In [5]:
with open(PROMPTS_DIR / "1_data_schema.txt", 'r', encoding='utf-8') as f:
    schema_prompt = f.read()

schema_prompt = schema_prompt.replace("{base_schema}", json.dumps(base_schema, indent=2, ensure_ascii=False))

print(f"正在生成 {DOMAIN} Schema...")
print("-" * 60)
schema_response = ""
for chunk in llm.llm.stream(schema_prompt):
    content = chunk.content
    schema_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

schema_text = extract_code_block(schema_response)
is_json, schema_data = verify_json_structure(schema_text)
if is_json:
    print(f"✓ JSON 结构验证通过，共 {len(schema_data)} 张表：")
    for t in schema_data.keys():
        print(f"  - {t}")
else:
    print(f"✗ JSON 验证失败：{schema_data}")

schema_path = GENERATED_DATA_DIR / "schema.json"
save_generated_file(str(schema_path), schema_text)
print(f"\n✓ Schema 已保存到: {schema_path}")

正在生成 HR Schema...
------------------------------------------------------------


```json
{
    "salary_structure": {
        "SSTR001": {
            "salary_structure_ref": "SSTR001",
            "employee_ref": "EMP001",
            "effective_date": "2025-01-15",
            "expiry_date": null,
            "base_salary": 30000.00,
            "currency": "CNY",
            "pay_frequency": "monthly",
            "status": "active",
            "approved_by_ref": "EMP000",
            "created_at": "2025-01-15 09:00:00",
            "updated_at": "2025-01-15 09:00:00"
        }
    },

    "salary_component": {
        "SCMP001": {
            "salary_component_ref": "SCMP001",
            "salary_structure_ref": "SSTR001",
            "component_type": "base_salary",
            "component_name": "基本工资",
            "amount": 30000.00,
            "currency": "CNY",
            "is_taxable": true,
            "is_recurring": true,
            "effective_date": "2025-01-15",
       

### 2. 生成 HR Data Models

In [9]:
with open(GENERATED_DATA_DIR / "schema.json", 'r', encoding='utf-8') as f:
    domain_schema = f.read()

with open(PROMPTS_DIR / "2_data_models.txt", 'r', encoding='utf-8') as f:
    model_prompt = f.read()

model_prompt = model_prompt.replace("{base_models}", base_models)
model_prompt = model_prompt.replace("{domain_schema}", domain_schema)

# print(model_prompt)
print(f"正在生成 {DOMAIN} Data Models...")
print("-" * 60)
model_response = ""
for chunk in llm.llm.stream(model_prompt):
    content = chunk.content
    model_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

model_code = extract_code_block(model_response)
is_valid, error = verify_python_syntax(model_code)
if is_valid:
    print("✓ Python 语法验证通过")
else:
    print(f"✗ Python 语法错误: {error}")

models_path = GENERATED_CODE_DIR / "models.py"
save_generated_file(str(models_path), model_code)
print(f"✓ Models 已保存到: {models_path}")

正在生成 HR Data Models...
------------------------------------------------------------


```python
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field

from bench.environment.db import DB
from generated.domains.BASE.models import OfficeDB


# --- Enums as Literal types ---

SalaryStructureStatus = Literal["active", "inactive", "expired"]
PayFrequency = Literal["monthly", "bi_weekly", "weekly"]
SalaryCurrency = Literal["CNY", "USD", "EUR", "GBP", "JPY", "KRW", "HKD"]

SalaryComponentType = Literal[
    "base_salary", "housing_allowance", "meal_allowance", "transport_allowance",
    "communication_allowance", "performance_bonus", "year_end_bonus",
    "overtime_pay", "equity_compensation", "sign_on_bonus", "other"
]

PayrollRecordStatus = Literal["draft", "calculated", "approved", "paid", "cancelled"]

PayrollLineItemCategory = Literal["earning", "deduction", "employer_contribution"]
PayrollLineItemType = Literal[
    "base_salary", "bonus", "allowan

### 3. Schema/Model 一致性验证

In [10]:
validation_prompt = f"""Schema:
{domain_schema}

Data model:
{model_code}

请检查以下三点，如果都满足则回答 "yes"，否则回答 "no" 并说明原因：
1. Data model 中的字段和类型与 Schema 定义一致
2. Data model 中的字段和类型在不同位置保持一致
3. 如果 Data model 更加合理，可以视为 yes

如果是 no，给出具体修改意见。
"""

print("正在验证 Schema 和 Model 的一致性...")
validation_response = ""
for chunk in llm.llm.stream(validation_prompt):
    validation_response += chunk.content
    print(chunk.content, end="", flush=True)
print()

正在验证 Schema 和 Model 的一致性...
我来逐一检查这三点。

## 检查过程

### 1. Data model 中的字段和类型与 Schema 定义一致

逐表对比：

- **salary_structure**: ✅ 一致
- **salary_component**: ✅ 一致
- **payroll_record**: ✅ 一致
- **payroll_line_item**: ✅ 一致
- **bank_account**: ✅ 一致
- **social_insurance**: ✅ 一致
- **housing_fund**: ✅ 一致
- **commercial_insurance**: ✅ 一致
- **employee_commercial_insurance**: ✅ 一致
- **benefit_plan**: ✅ 一致
- **employee_benefit_enrollment**: ✅ 一致
- **work_schedule**: ✅ 一致
- **employee_work_schedule**: ✅ 一致
- **holiday_calendar**: ✅ 一致
- **leave_policy**: ✅ 一致
- **leave_balance**: ✅ 一致
- **leave_request**: ✅ 一致
- **overtime_request**: ✅ 一致
- **contract**: ✅ 一致
- **employee_change**: ✅ 一致
- **approval_flow**: ✅ 一致
- **approval_step**: ✅ 一致
- **job_requisition**: ✅ 一致
- **candidate**: ✅ 一致
- **job_application**: ✅ 一致
- **referral**: ✅ 一致
- **interview**: ✅ 一致
- **offer**: ✅ 一致
- **training_program**: ✅ 一致
- **training_enrollment**: ✅ 一致
- **probation_review**: ✅ 一致
- **performance_cycle**: ✅ 一致
- **performanc

In [ ]:
# 若验证结果为 no，执行此 cell 重新生成 model；否则跳过
regenerate_prompt = f"""当前的 Schema 为：
{domain_schema}

当前的 Model 为：
{model_code}

校验结果：
{validation_response}

请根据校验结果修改 model，保持 Python 语法正确。
"""

fix_response = llm.generate(regenerate_prompt)
model_code = extract_code_block(fix_response)

is_valid, error = verify_python_syntax(model_code)
if is_valid:
    print("✓ Python 语法验证通过")
    models_path = GENERATED_CODE_DIR / "models.py"
    save_generated_file(str(models_path), model_code)
    print(f"✓ 修正后 Models 已保存到: {models_path}")
else:
    print(f"✗ Python 语法错误: {error}")

### 4. 生成数据脚本（Faker，主流程）

分析 HR Models 的外键依赖，生成 Faker 脚本（固定 seed，可复现），然后执行脚本产生 db.json。

In [ ]:
# 从已保存的 models.py 加载（确保使用最新版本）
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

# 分析 HR 域的外键依赖（不含 BASE 表）
ref_map = parse_ref_fields(model_code)
all_tables = list(ref_map.keys())
table_order = topological_sort(all_tables, ref_map)
dependency_info = {
    table: [(field, target) for field, target in refs]
    for table, refs in ref_map.items()
}

print(f"HR 域表生成顺序（拓扑排序，共 {len(table_order)} 张表）：")
for i, t in enumerate(table_order, 1):
    print(f"  {i:2d}. {t}")
print(f"\nFK 依赖映射：")
for t, deps in dependency_info.items():
    if deps:
        print(f"  {t}: {deps}")

# 加载并填充 prompt（4 个占位符全部填入）
with open(PROMPTS_DIR / "3-1_data_script.txt", 'r', encoding='utf-8') as f:
    script_prompt = f.read()

script_prompt = script_prompt.replace("{base_data}", base_data_raw)
script_prompt = script_prompt.replace("{base_gen_data}", base_gen_data_code)
script_prompt = script_prompt.replace("{model_code}", model_code)
script_prompt = script_prompt.replace("{table_order}", str(table_order))
script_prompt = script_prompt.replace("{dependency_info}", str(dependency_info))


local_llm = LLMClient("gpt-5.4-0305-global", 128000)
print(f"\n正在生成 {DOMAIN} 数据脚本...")
print("-" * 60)
full_response = ""
for chunk in local_llm.llm.stream(script_prompt):
    content = chunk.content
    full_response += content
    print(content, end="", flush=True)
print(f"\n{'-' * 60}")

script_code = extract_code_block(full_response)
ok, err = verify_python_syntax(script_code)
if not ok:
    print(f"❌ 语法错误：{err}")
else:
    print("✓ Python 语法验证通过")

script_path = GENERATED_CODE_DIR / "gen_data.py"
save_generated_file(str(script_path), script_code)
print(f"✓ 数据脚本已保存到: {script_path}")

HR 域表生成顺序（拓扑排序，共 44 张表）：
   1. salarystructures
   2. salarycomponents
   3. payrollrecords
   4. payrolllineitems
   5. bankaccounts
   6. socialinsurances
   7. housingfunds
   8. commercialinsurances
   9. employeecommercialinsurances
  10. benefitplans
  11. employeebenefitenrollments
  12. workschedules
  13. employeeworkschedules
  14. holidaycalendars
  15. leavepolicies
  16. leavebalances
  17. leaverequests
  18. overtimerequests
  19. contracts
  20. employeechanges
  21. approvalflows
  22. approvalsteps
  23. jobrequisitions
  24. candidates
  25. interviews
  26. trainingprograms
  27. trainingenrollments
  28. performancecycles
  29. performancereviews
  30. equitygrants
  31. equityvestingevents
  32. professionaltitles
  33. talentsubsidies
  34. employeerecords
  35. visarecords
  36. taxdeclarations
  37. taxspecialdeductions
  38. compliancereports
  39. certificaterequests
  40. hrservicerequests
  41. probationreviews
  42. referrals
  43. offers
  44. jobapplicat

In [11]:
# 执行数据生成脚本，捕获 JSON 输出
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()
script_path = GENERATED_CODE_DIR / "gen_data.py"
ref_map = parse_ref_fields(model_code)
print("正在执行数据生成脚本...")
result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True, text=True, timeout=120,
    cwd=str(Path(".").resolve())
)

if result.returncode != 0:
    print(f"❌ 脚本执行失败：\n{result.stderr[-3000:]}")
else:
    print("✓ 脚本执行成功")
    is_json, db_data = verify_json_structure(result.stdout)
    if not is_json:
        print(f"❌ 输出 JSON 解析失败：{db_data}")
    else:
        print(f"✓ JSON 解析成功，共 {len(db_data)} 张表")
        for tname, tdata in db_data.items():
            if isinstance(tdata, dict):
                print(f"  - {tname}: {len(tdata)} 条")

        # FK 完整性验证（HR 域 + BASE 域合并验证）
        merged_data = {**base_data, **db_data}
        fk_ok, fk_errors = validate_foreign_keys(merged_data, ref_map)
        if fk_ok:
            print("✓ 外键完整性验证通过")
        else:
            print(f"⚠️  外键错误 {len(fk_errors)} 处（前5条）：")
            for e in fk_errors[:5]:
                print(f"  - {e}")

        db_path = GENERATED_DATA_DIR / "db.json"
        save_generated_file(str(db_path), result.stdout)
        print(f"\n✓ HR Database 已保存到: {db_path}")

正在执行数据生成脚本...
✓ 脚本执行成功
✓ JSON 解析成功，共 44 张表
  - salary_structure: 225 条
  - salary_component: 1488 条
  - payroll_record: 225 条
  - payroll_line_item: 2475 条
  - bank_account: 75 条
  - social_insurance: 375 条
  - housing_fund: 97 条
  - commercial_insurance: 12 条
  - employee_commercial_insurance: 115 条
  - benefit_plan: 12 条
  - employee_benefit_enrollment: 112 条
  - work_schedule: 10 条
  - employee_work_schedule: 75 条
  - holiday_calendar: 20 条
  - leave_policy: 12 条
  - leave_balance: 450 条
  - leave_request: 225 条
  - overtime_request: 225 条
  - contract: 98 条
  - employee_change: 225 条
  - approval_flow: 647 条
  - approval_step: 1527 条
  - job_requisition: 75 条
  - candidate: 225 条
  - interview: 335 条
  - training_program: 12 条
  - training_enrollment: 112 条
  - performance_cycle: 10 条
  - performance_review: 114 条
  - equity_grant: 19 条
  - equity_vesting_event: 62 条
  - professional_title: 99 条
  - talent_subsidy: 34 条
  - employee_record: 225 条
  - visa_record: 20 条
  - tax_decla

In [4]:
# 用 Pydantic 模型做最终验证
import json as _json
import sys as _sys
_sys.path.insert(0, str(Path(".").resolve()))
from generated.domains.HR.models import HRDB

with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    db_raw = _json.load(f)

# HRDB 继承 OfficeDB（BASE 域），验证时需合并 BASE 数据
merged_raw = {**base_data, **db_raw}

try:
    office_db = HRDB.model_validate(merged_raw)
    print("✓ OfficeDB 模型验证通过")
    print(office_db.get_statistics())
except Exception as e:
    print(f"❌ 模型验证失败：{e}")

✓ OfficeDB 模型验证通过
{'num_users': 75, 'num_employees': 75, 'num_departments': 12, 'num_teams': 22, 'num_roles': 8, 'num_permissions': 29, 'num_salary_structures': 225, 'num_salary_components': 1488, 'num_payroll_records': 225, 'num_payroll_line_items': 2475, 'num_bank_accounts': 75, 'num_social_insurances': 375, 'num_housing_funds': 97, 'num_commercial_insurances': 12, 'num_employee_commercial_insurances': 115, 'num_benefit_plans': 12, 'num_employee_benefit_enrollments': 112, 'num_work_schedules': 10, 'num_employee_work_schedules': 75, 'num_holiday_calendars': 20, 'num_leave_policies': 12, 'num_leave_balances': 450, 'num_leave_requests': 225, 'num_overtime_requests': 225, 'num_contracts': 98, 'num_employee_changes': 225, 'num_approval_flows': 647, 'num_approval_steps': 1527, 'num_job_requisitions': 75, 'num_candidates': 225, 'num_job_applications': 225, 'num_referrals': 67, 'num_interviews': 335, 'num_offers': 58, 'num_training_programs': 12, 'num_training_enrollments': 112, 'num_probati

### 5. 数据验证（覆盖度统计）

In [7]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()
ref_map = parse_ref_fields(model_code)
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    hr_db_data = _json.load(f)

# 用 BASE 数据的 employees 作为覆盖度参照
coverage_stats = calculate_coverage_statistics(hr_db_data, base_data, ref_map)
report = format_validation_report(
    fk_valid=True, fk_errors=[],
    coverage_stats=coverage_stats,
    min_records_per_table=5,
    min_records_per_employee=3,
)
print(report)
print(f"\n总记录数：{coverage_stats['total_records']}")

✓ 所有验证通过

总记录数：11381


### 6. 生成 Tools

In [18]:
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    model_code = f.read()

with open(PROMPTS_DIR / "4_module_tools.txt", 'r', encoding='utf-8') as f:
    tools_prompt = f.read()

# HR 域功能描述 —— 按数据库表逐条列出，LLM 必须全部实现
hr_function_description = """
Below is a PER-TABLE mandatory checklist. You MUST implement every function listed.
Use the EXACT function names specified. Do NOT skip any item.

--- BASE tools (already in OfficeTools, DO NOT reimplement) ---
SKIP: employee, department, team, position, role, permission, mentorship, org_config, notification
  These are all handled by the inherited OfficeTools class.

--- 薪酬 ---
TABLE: salary_structure
  READ:  get_salary_structure(salary_structure_ref: str) -> SalaryStructure
  READ:  get_employee_salary_structures(employee_ref: str) -> List[SalaryStructure]  # sorted by effective_date desc

TABLE: salary_component
  READ:  get_salary_components(salary_structure_ref: str) -> List[SalaryComponent]

TABLE: payroll_record
  READ:  get_payroll_record(payroll_record_ref: str) -> PayrollRecord
  READ:  get_employee_payroll_records(employee_ref: str) -> List[PayrollRecord]  # sorted by pay_period_start desc

TABLE: payroll_line_item
  READ:  get_payroll_line_items(payroll_record_ref: str) -> List[PayrollLineItem]

TABLE: bank_account
  READ:  get_bank_account(bank_account_ref: str) -> BankAccount
  READ:  get_employee_bank_accounts(employee_ref: str) -> List[BankAccount]
  WRITE: add_bank_account(employee_ref, bank_name, branch_name, account_number_masked, account_holder_name, is_primary, effective_date) -> BankAccount  # if is_primary, demote others
  WRITE: set_primary_bank_account(bank_account_ref) -> BankAccount  # demote other accounts of same employee

--- 社保/公积金/保险/福利 ---
TABLE: social_insurance
  READ:  get_social_insurance(social_insurance_ref: str) -> SocialInsurance
  READ:  get_employee_social_insurances(employee_ref: str) -> List[SocialInsurance]

TABLE: housing_fund
  READ:  get_housing_fund(housing_fund_ref: str) -> HousingFund
  READ:  get_employee_housing_funds(employee_ref: str) -> List[HousingFund]

TABLE: commercial_insurance
  READ:  get_commercial_insurance(commercial_insurance_ref: str) -> CommercialInsurance
  READ:  list_commercial_insurances() -> List[CommercialInsurance]

TABLE: employee_commercial_insurance
  READ:  get_employee_commercial_insurance(employee_commercial_insurance_ref: str) -> EmployeeCommercialInsurance
  READ:  get_employee_commercial_insurances(employee_ref: str) -> List[EmployeeCommercialInsurance]

TABLE: benefit_plan
  READ:  get_benefit_plan(benefit_plan_ref: str) -> BenefitPlan
  READ:  list_benefit_plans() -> List[BenefitPlan]

TABLE: employee_benefit_enrollment
  READ:  get_employee_benefit_enrollment(employee_benefit_enrollment_ref: str) -> EmployeeBenefitEnrollment
  READ:  get_employee_benefit_enrollments(employee_ref: str) -> List[EmployeeBenefitEnrollment]

--- 工作日程 ---
TABLE: work_schedule
  READ:  get_work_schedule(work_schedule_ref: str) -> WorkSchedule
  READ:  list_work_schedules() -> List[WorkSchedule]

TABLE: employee_work_schedule
  READ:  get_employee_work_schedule(employee_work_schedule_ref: str) -> EmployeeWorkSchedule
  READ:  get_employee_work_schedules(employee_ref: str) -> List[EmployeeWorkSchedule]

TABLE: holiday_calendar
  READ:  get_holiday_calendar(holiday_calendar_ref: str) -> HolidayCalendar
  READ:  list_holiday_calendars(year: Optional[int] = None) -> List[HolidayCalendar]

--- 请假 ---
TABLE: leave_policy
  READ:  get_leave_policy(leave_policy_ref: str) -> LeavePolicy
  READ:  list_leave_policies() -> List[LeavePolicy]

TABLE: leave_balance
  READ:  get_employee_leave_balances(employee_ref: str, year: int) -> List[LeaveBalance]

TABLE: leave_request
  READ:  get_leave_request(leave_request_ref: str) -> LeaveRequest
  READ:  get_employee_leave_requests(employee_ref: str) -> List[LeaveRequest]
  READ:  get_pending_leave_approvals(approver_employee_ref: str) -> List[LeaveRequest]
  WRITE: submit_leave_request(employee_ref, leave_policy_ref, start_datetime, end_datetime, duration_days, reason, handover_employee_ref=None) -> LeaveRequest  # check leave balance
  WRITE: cancel_leave_request(leave_request_ref, reason) -> LeaveRequest  # only pending/draft
  WRITE: approve_leave_request(leave_request_ref, approver_employee_ref, comment=None) -> LeaveRequest  # deduct leave balance
  WRITE: reject_leave_request(leave_request_ref, approver_employee_ref, comment) -> LeaveRequest

--- 加班 ---
TABLE: overtime_request
  READ:  get_overtime_request(overtime_request_ref: str) -> OvertimeRequest
  READ:  get_employee_overtime_requests(employee_ref: str) -> List[OvertimeRequest]
  READ:  get_pending_overtime_approvals(approver_employee_ref: str) -> List[OvertimeRequest]
  WRITE: submit_overtime_request(employee_ref, overtime_date, start_time, end_time, duration_hours, overtime_type, compensation_type, reason) -> OvertimeRequest
  WRITE: approve_overtime_request(overtime_request_ref, approver_employee_ref, comment=None) -> OvertimeRequest
  WRITE: reject_overtime_request(overtime_request_ref, approver_employee_ref, comment) -> OvertimeRequest
  WRITE: cancel_overtime_request(overtime_request_ref, reason) -> OvertimeRequest  # only pending/draft

--- 合同 ---
TABLE: contract
  READ:  get_contract(contract_ref: str) -> Contract
  READ:  get_employee_contracts(employee_ref: str) -> List[Contract]
  READ:  get_expiring_contracts(before_date: str) -> List[Contract]  # active contracts with end_date <= before_date
  WRITE: renew_contract(contract_ref, new_end_date, notes='') -> Contract  # renewal_count+1, status='active'
  WRITE: terminate_contract(contract_ref, termination_date, reason, notes='') -> Contract  # only active, status='terminated'

--- 员工变动（入职/离职/晋升/调岗/薪调） ---
TABLE: employee_change
  READ:  get_employee_change(employee_change_ref: str) -> EmployeeChange
  READ:  get_employee_changes(employee_ref: str, change_type: Optional[str] = None) -> List[EmployeeChange]
  READ:  get_onboarding_progress(employee_ref: str) -> Dict  # change_type='onboarding' + contracts + training
  READ:  get_offboarding_progress(employee_ref: str) -> Dict  # change_type='offboarding' + contracts
  WRITE: update_onboarding_step_status(employee_change_ref, new_status, notes=None) -> EmployeeChange
  WRITE: initiate_offboarding(employee_ref, effective_date, reason, notes='') -> EmployeeChange
  WRITE: update_offboarding_step_status(employee_change_ref, new_status, notes=None) -> EmployeeChange
  WRITE: initiate_transfer(employee_ref, effective_date, to_department_ref, to_position_ref, to_manager_ref, reason, notes='') -> EmployeeChange  # change_type='transfer'
  WRITE: initiate_promotion(employee_ref, effective_date, to_position_ref, to_salary_structure_ref, reason, notes='') -> EmployeeChange  # change_type='promotion'

--- 审批流 ---
TABLE: approval_flow
  READ:  get_approval_flow(approval_flow_ref: str) -> ApprovalFlow

TABLE: approval_step
  READ:  get_approval_steps(approval_flow_ref: str) -> List[ApprovalStep]  # sorted by step_order

--- 招聘 ---
TABLE: job_requisition
  READ:  get_job_requisition(job_requisition_ref: str) -> JobRequisition
  READ:  list_job_requisitions(status: Optional[str] = None) -> List[JobRequisition]

TABLE: candidate
  READ:  get_candidate(candidate_ref: str) -> Candidate

TABLE: job_application
  READ:  get_job_application(job_application_ref: str) -> JobApplication
  READ:  get_applications_by_requisition(job_requisition_ref: str) -> List[JobApplication]
  READ:  get_applications_by_candidate(candidate_ref: str) -> List[JobApplication]
  WRITE: advance_application_stage(job_application_ref, new_stage, notes='') -> JobApplication  # screening/phone_screen/interview/assessment/offer/hired/rejected/withdrawn

TABLE: referral
  READ:  get_referral(referral_ref: str) -> Referral
  READ:  get_employee_referrals(referrer_employee_ref: str) -> List[Referral]

TABLE: interview
  READ:  get_interview(interview_ref: str) -> Interview
  READ:  get_interviews_by_application(job_application_ref: str) -> List[Interview]
  WRITE: schedule_interview(job_application_ref, interviewer_employee_ref, interview_round, interview_type, scheduled_start, scheduled_end, location=None, meeting_link=None) -> Interview
  WRITE: submit_interview_feedback(interview_ref, rating, feedback, result) -> Interview  # rating 1-5, result: pass/fail/strong_pass/borderline/no_show

TABLE: offer
  READ:  get_offer(offer_ref: str) -> Offer
  READ:  get_offers_by_application(job_application_ref: str) -> List[Offer]
  WRITE: extend_offer(job_application_ref, position_ref, department_ref, offered_salary, currency, offered_position_title, offered_worker_type, expected_hire_date, offer_valid_until, sign_on_bonus=0.0, special_terms=None) -> Offer
  WRITE: update_offer_status(offer_ref, new_status) -> Offer  # issued/approved/accepted/rejected/withdrawn

--- 培训 ---
TABLE: training_program
  READ:  get_training_program(training_program_ref: str) -> TrainingProgram
  READ:  list_training_programs(status: Optional[str] = None) -> List[TrainingProgram]

TABLE: training_enrollment
  READ:  get_training_enrollment(training_enrollment_ref: str) -> TrainingEnrollment
  READ:  get_employee_training_enrollments(employee_ref: str) -> List[TrainingEnrollment]
  WRITE: enroll_training(employee_ref, training_program_ref) -> TrainingEnrollment  # program must be active, no duplicate active enrollment
  WRITE: update_training_enrollment_completion(training_enrollment_ref, progress_pct, score=None, result=None) -> TrainingEnrollment

--- 试用期 ---
TABLE: probation_review
  READ:  get_probation_review(probation_review_ref: str) -> ProbationReview
  READ:  get_employee_probation_reviews(employee_ref: str) -> List[ProbationReview]
  WRITE: submit_probation_review(probation_review_ref, rating, strengths, improvements, recommendation) -> ProbationReview  # status must be draft/pending -> completed

--- 绩效 ---
TABLE: performance_cycle
  READ:  get_performance_cycle(performance_cycle_ref: str) -> PerformanceCycle
  READ:  list_performance_cycles() -> List[PerformanceCycle]

TABLE: performance_review
  READ:  get_performance_review(performance_review_ref: str) -> PerformanceReview
  READ:  get_employee_performance_reviews(employee_ref: str) -> List[PerformanceReview]
  WRITE: submit_performance_self_review(performance_review_ref, self_rating, self_summary, goals_achieved) -> PerformanceReview  # status must be draft/self_review
  WRITE: submit_performance_manager_review(performance_review_ref, reviewer_employee_ref, final_rating, manager_comment, goals_for_next_period=None) -> PerformanceReview  # status must be manager_review

--- 股权 ---
TABLE: equity_grant
  READ:  get_equity_grant(equity_grant_ref: str) -> EquityGrant
  READ:  get_employee_equity_grants(employee_ref: str) -> List[EquityGrant]

TABLE: equity_vesting_event
  READ:  get_equity_vesting_event(equity_vesting_event_ref: str) -> EquityVestingEvent
  READ:  get_grant_vesting_events(equity_grant_ref: str) -> List[EquityVestingEvent]

--- 职称/补贴/档案/签证 ---
TABLE: professional_title
  READ:  get_professional_title(professional_title_ref: str) -> ProfessionalTitle
  READ:  get_employee_professional_titles(employee_ref: str) -> List[ProfessionalTitle]

TABLE: talent_subsidy
  READ:  get_talent_subsidy(talent_subsidy_ref: str) -> TalentSubsidy
  READ:  get_employee_talent_subsidies(employee_ref: str) -> List[TalentSubsidy]

TABLE: employee_record
  READ:  get_employee_record(employee_record_ref: str) -> EmployeeRecord
  READ:  get_employee_records(employee_ref: str) -> List[EmployeeRecord]

TABLE: visa_record
  READ:  get_visa_record(visa_record_ref: str) -> VisaRecord
  READ:  get_employee_visa_records(employee_ref: str) -> List[VisaRecord]

--- 税务 ---
TABLE: tax_declaration
  READ:  get_tax_declaration(tax_declaration_ref: str) -> TaxDeclaration
  READ:  get_employee_tax_declarations(employee_ref: str) -> List[TaxDeclaration]

TABLE: tax_special_deduction
  READ:  get_tax_special_deduction(tax_special_deduction_ref: str) -> TaxSpecialDeduction
  READ:  get_employee_tax_special_deductions(employee_ref: str) -> List[TaxSpecialDeduction]

--- 合规 ---
TABLE: compliance_report
  READ:  get_compliance_report(compliance_report_ref: str) -> ComplianceReport
  READ:  list_compliance_reports(status: Optional[str] = None) -> List[ComplianceReport]

--- 证明开具 ---
TABLE: certificate_request
  READ:  get_certificate_request(certificate_request_ref: str) -> CertificateRequest
  READ:  get_employee_certificate_requests(employee_ref: str) -> List[CertificateRequest]
  WRITE: submit_certificate_request(employee_ref, certificate_type, purpose, language='zh-CN', additional_info=None) -> CertificateRequest
  WRITE: cancel_certificate_request(certificate_request_ref, reason) -> CertificateRequest  # only pending/draft

--- HR 服务请求 ---
TABLE: hr_service_request
  READ:  get_hr_service_request(hr_service_request_ref: str) -> HRServiceRequest
  READ:  get_employee_hr_service_requests(employee_ref: str) -> List[HRServiceRequest]
  WRITE: submit_hr_service_request(requester_employee_ref, request_type, subject, description, priority='normal') -> HRServiceRequest
  WRITE: update_hr_service_request_status(hr_service_request_ref, new_status, resolution_notes=None) -> HRServiceRequest
  WRITE: cancel_hr_service_request(hr_service_request_ref, reason) -> HRServiceRequest  # only open/in_progress
"""

with open(Path("generated/domains/BASE/tools.py"), 'r', encoding='utf-8') as f:
    base_tools_code = f.read()

tools_prompt = tools_prompt.replace("{domain}", DOMAIN)
tools_prompt = tools_prompt.replace("{base_tools}", base_tools_code)
tools_prompt = tools_prompt.replace("{model_code}", model_code)
tools_prompt = tools_prompt.replace("{function_description}", hr_function_description)

print(f"正在生成 {DOMAIN} Tools...")
tools_response = ""
for chunk in llm.llm.stream(tools_prompt):
    content = chunk.content
    tools_response += content
    print(content, end="", flush=True)

tools_code = extract_code_block(tools_response)
is_valid, error = verify_python_syntax(tools_code)
if is_valid:
    print("\n✓ Python 语法验证通过")
else:
    print(f"\n✗ Python 语法错误: {error}")

tools_path = GENERATED_CODE_DIR / "tools.py"
save_generated_file(str(tools_path), tools_code)
print(f"✓ Tools 已保存到: {tools_path}")

正在生成 HR Tools...


```python
"""Toolkit for the HR domain, extending OfficeTools with HR-specific operations."""

from typing import Any, Dict, List, Optional
import json
from datetime import datetime, timezone

from bench.environment.toolkit import ToolType, is_tool
from generated.domains.BASE.tools import OfficeTools
from generated.domains.hr.data_model import (
    ApprovalFlow,
    ApprovalStep,
    BankAccount,
    BenefitPlan,
    CertificateRequest,
    Candidate,
    CommercialInsurance,
    ComplianceReport,
    Contract,
    EmployeeBenefitEnrollment,
    EmployeeChange,
    EmployeeCommercialInsurance,
    EmployeeRecord,
    EmployeeWorkSchedule,
    EquityGrant,
    EquityVestingEvent,
    HolidayCalendar,
    HousingFund,
    HRServiceRequest,
    HRDB,
    Interview,
    JobApplication,
    JobRequisition,
    LeaveBalance,
    LeavePolicy,
    LeaveRequest,
    Offer,
    OvertimeRequest,
    PayrollLineItem,
    PayrollRecord,
    PerformanceCycle,
    PerformanceRevie

### 7. 生成 Tasks（结构驱动，无 purpose）

In [ ]:
from synthesis.task_gen import TaskGenerationPipeline

with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()
with open(GENERATED_CODE_DIR / "models.py", 'r', encoding='utf-8') as f:
    models_code = f.read()
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    domain_db = _json.load(f)

pipeline = TaskGenerationPipeline(
    domain=DOMAIN,
    tools_code=tools_code,
    models_code=models_code,
    domain_db=domain_db,
    base_db=base_data,
    llm=llm,
)
tasks = pipeline.run()

tasks_path = GENERATED_DATA_DIR / "tasks.json"
with open(tasks_path, "w", encoding="utf-8") as f:
    _json.dump(tasks, f, ensure_ascii=False, indent=2)
print(f"已生成 {len(tasks)} 个任务 → {tasks_path}")


### 8. 任务质量验证

结构覆盖度、WRITE 工具覆盖率、员工均匀度、难度分布、ID 合法性检查。

In [7]:
import re as _re
from synthesis.task_gen import verify_tasks, ToolDAGParser

with open(GENERATED_DATA_DIR / "tasks.json", 'r', encoding='utf-8') as f:
    tasks = _json.load(f)
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    domain_db = _json.load(f)
with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()

dag = ToolDAGParser().parse(tools_code)
metrics = verify_tasks(tasks, dag, base_data)

# 合并所有合法 ID（base + domain）用于 ID 合法性检查
all_valid_ids = set()
for table_data in {**base_data, **domain_db}.values():
    if isinstance(table_data, dict):
        all_valid_ids.update(table_data.keys())

# ---- 1. ID 合法性检查 ----
id_errors = []
for task in tasks:
    task_id = task.get('id', '?')
    for action in task.get('evaluation_criteria', {}).get('actions', []):
        for arg_key, arg_val in action.get('arguments', {}).items():
            if arg_key.endswith('_ref') and isinstance(arg_val, str):
                if arg_val not in all_valid_ids:
                    id_errors.append(f"任务 {task_id} | 工具 {action['name']} | {arg_key}={arg_val} → 不存在于任何表")
print("=" * 60)
print("【ID 合法性检查】")
if not id_errors:
    print("✓ 所有 *_ref 参数均为合法 ID")
else:
    print(f"❌ 发现 {len(id_errors)} 处无效 ID：")
    for e in id_errors[:20]:
        print(f"  - {e}")

# ---- 2. 结构覆盖度 ----
print("\n=== Pattern 分布 ===")
for pattern, count in sorted(metrics.get("pattern_coverage", {}).items()):
    print(f"  {pattern}: {count}")

# ---- 3. WRITE 工具覆盖率 ----
wc = metrics.get("write_tool_coverage")
if wc is not None:
    print(f"\nWRITE 工具覆盖率: {wc:.1%} {'✓' if wc >= 0.8 else '⚠️'} （目标 ≥ 80%）")

# ---- 4. 员工均匀度 ----
eu = metrics.get("employee_uniformity")
if eu:
    print(f"员工均匀度 (CV): {eu['cv']:.2f} {'✓' if eu['cv'] <= 0.3 else '⚠️'} （目标 ≤ 0.3）")

# ---- 5. WRITE 任务占比 ----
wr = metrics.get("write_task_ratio")
if wr is not None:
    print(f"WRITE 任务占比: {wr:.1%} {'✓' if wr >= 0.6 else '⚠️'} （目标 ≥ 60%）")

# ---- 6. 难度分布 ----
print("\n=== 难度分布 ===")
total = len(tasks)
for level in sorted(metrics.get("difficulty_distribution", {}).keys()):
    count = metrics["difficulty_distribution"][level]
    print(f"  {level}: {count} ({count/total:.1%})")
print(f"  总计: {total}")
purposes = ['查最新薪资结构', '查本月工资单', '查当前银行卡', '查当前社保', '查当前公积金', '查当前商保', '查当前福利', '查当前班制', '查进行中合同', '查待办请假', '查待办加班', '查候选人申请', '查岗位面试', '查员工股权', '查个税申报', '查专项附加扣除', '查证明确认', '查HR工单', '查试用评估', '查绩效评估', '核对工资构成', '比对工资与结构', '核银行打款信息', '汇总社保公积金', '汇总商保福利', '核班制节假日', '核请假余额', '核加班申请历史', '看入职进度', '看离职进度', '查审批链路', '汇总岗位候选人', '看候选人面试报价', '查推荐转化', '查培训完成情况', '核试用与培训', '看绩效周期状态', '查股权归属计划', '看职称补贴匹配', '看档案签证情况', '核税务扣除', '看证明办理状态', '看HR工单进展', '查合同与变动', '交年假申请', '交病假申请', '交事假申请', '交婚假申请', '交陪产假申请', '交调休假申请', '交跨天请假', '交带交接人请假', '交英文证明', '交收入证明', '交社保证明', '交公积金证明', '交其他证明', '提HR政策咨询', '提系统异常单', '提入职支持单', '提离职支持单', '提交加班申请', '提交工作日加班', '提交节假日加班', '混合补偿加班', '提交绩效自评', '提交经理评语', '更新培训进度', '培训记为完成', '取消待审请假', '取消待审加班', '取消证明申请', '取消HR工单', '离职发起申请', '更新入职步骤', '更新离职步骤', '更新工单状态', '补提在职证明', '再次提收入证明', '查后提咨询单', '开紧急工单', '申请推荐信', '提交文档请求', '问后发投诉单', '更新培训成绩', '续提离职证明', '补提加班申请', '补提请假申请', '经理批准请假', '经理驳回请假', '经理批准加班', '经理驳回加班', 'HR发起调岗', 'HR发起晋升', 'HR发起离职', '完成入职步骤', '完成离职步骤', '解决HR工单', '关闭HR工单', '批准多级请假', '批准多级加班', '确认后撤请假', '确认后撤加班', '确认后撤证明', '确认后撤工单', '结训标记通过', '结训标记未过', '自评前先核状态', '经理评前核状态', '提假前核余额', '提假前核政策', '提证前判类型', '提工单前去重', '核节日后报加班', '先核工单再更新', '变更入职为待批', '变更离职为作废', '审后核余额变化', '调岗前看历史变动', '晋升前看薪资方案', '离职前看合同到期', '批加班前看历史', '批假前看历史请假', '提工单并设高优先', '提证明前查历史', '提入职支持并附进度', '提离职支持并附进度', '核培训再完结', '查绩效后提自评', '查审批流后拒假', '查审批流后拒班', '重复假单前去重', '重复班单前去重', '批假并留意见', '批班并留意见', '提交英文收入证', '提交英文在职证', '查合同后提离职', '查记录后改工单', '看周期后经理评', '转岗前核经理存在', '晋升前核岗位存在', '核余额后批假', '核节假日后批班', '取消已批请假', '取消已批加班', '取消已出证明', '取消已关闭工单', '余额不足提请假', '无权审批请假', '无权审批加班', '无理由拒请假', '无理由拒加班', '错阶段交自评', '错阶段交经理评', '改已取消培训', '错类型更入职', '错类型更离职', '不存在交接人请假', '不存在经理调岗', '不存在岗位晋升', '不存在薪资方案晋升', '重复发起离职', '重复证明申请', '重复HR工单', '重叠时段请假', '重复时段加班', '错误证明类型', '错误语言代码', '错误工单类型', '错误工单优先级', '错误加班类型', '错误补偿方式', '非法培训结果', '非法培训进度', '非法工单状态', '非法入职状态', '非法离职状态', '批已拒请假', '批已拒加班', '拒已批请假', '拒已批加班', '已关闭工单改处理中', '已取消工单改解决', '无确认直接写假单', '无确认直接写加班', '无确认直接开证明', '无确认直接开工单', '相似姓名批假', '相似姓名提证明', '相似姓名调岗', '相似姓名离职', '相似姓名晋升', '错人代批请假', '错人代批加班', '错人改他人绩效', '错人改他人培训', '错人撤他人工单', '错人撤他人证明', '错人发起晋升', '错人发起离职', '假种与余额冲突', '日期与天数冲突', '加班时长冲突', '节日类型冲突', '重复离职证明误用', '无待办却审批', '无审批流强行批', '工单解决缺说明', '离职前合同已失效', '调岗目标同当前', '晋升目标未变化', '培训未报名更新', '绩效对象不明', '工资核对后乱改卡', '股权查询后求写入', '合同查询后求续签', '社保查询后求补缴', '公积金查询后求修改', '税务查询后求更正', '签证记录后求续签', '档案记录后求删除', '福利记录后求变更', '班制记录后求改班', '商保记录后求增员', '招聘需求后求关闭', '候选人后求发offer', '面试后求改时间', '推荐记录后求发奖', '补贴记录后求发放', '职称记录后求新增', '员工记录后求修正', '合规报告后求关闭', '工资核对后改薪', '假单已批再重批', '班单已批再重批', '错对象交经理评', '错对象交自评', '已完成培训再更新', '离职员工提在职证', '在职员工提离职证', '合同未到期硬离职', '审批后无余额记录', '错误年度查余额', '工单重复关闭', '工单取消后重取消', '证明取消后重取消', '请假取消后重取消', '加班取消后重取消', '错误对象解决工单', '错误对象更新培训', '错误对象更新变动', '误把工单当证明', '误把证明当工单', '工单与历史冲突', '证明与历史冲突', '请假与历史冲突', '加班与历史冲突', '审批意见缺身份', '离职支持错分类', '入职支持错分类', '证明语言用途矛盾', '跨年请假余额不足', '假单审批后越权取消', '班单审批后越权取消', '错误对象取消证明', '错误对象取消工单', '错误对象提证明', '错误对象开工单', '越权查看后写操作', '审批前无明确单据', '工单更新目标不明', '证明取消目标不明', '请假取消目标不明', '加班取消目标不明', '培训更新目标不明', '绩效提交目标不明', '离职发起对象不明', '调岗目标信息缺失', '晋升目标信息缺失', '证明用途缺失', '工单描述缺失', '请假原因缺失', '加班原因缺失', '离职原因缺失', '调岗原因缺失', '晋升原因缺失', '绩效内容过短', '工单已解决再取消', '证明已完成再取消', '请假草稿外强撤', '加班草稿外强撤', '审批后错误人操作', '跨模块错误推断', '培训完成缺结果', '经理评缺评语', '自评缺总结', '离职支持已结案重开', '入职支持重复开单', '审批链无当前步批准', '审批链无当前步拒绝', '错误年份工资核对', '离职后再提加班', '离职后再提请假', '新员工未有记录硬提绩效', '无合同却发离职', '工单转状态跳步', '培训进度倒退异常', '审批后余额成负数风险', '同名候选人混淆', '同名员工合同离职混淆']  # 也可手动指定子集

all_tasks = []
MAX_RETRIES = 3
local_llm = LLMClient("gpt-5.4-0305-global", 128000)
for purpose in purposes:
    print(f"\n{'='*60}")
    print(f"正在生成 purpose: {purpose}")

    # 依赖感知采样：只注入与 purpose 相关表的记录，确保 employee_ref 可对齐
    domain_sample, seed_emps = filter_db_for_purpose(domain_db, purpose)
    base_sample = {**base_sample_base}
    if seed_emps and "employee" in base_data:
        base_sample["employee"] = {
            k: v for k, v in base_data["employee"].items() if k in seed_emps
        }

    task_prompt = task_prompt_template
    task_prompt = task_prompt.replace("{tool_code}", tools_code)
    task_prompt = task_prompt.replace("{base_content}", _json.dumps(base_sample, ensure_ascii=False, indent=2))
    task_prompt = task_prompt.replace("{domain_content}", _json.dumps(domain_sample, ensure_ascii=False, indent=2))
    task_prompt = task_prompt.replace("{purpose}", purpose)

    for attempt in range(MAX_RETRIES):
        task_response = local_llm.generate(task_prompt)
        task_text = extract_code_block(task_response)
        is_valid, task_data = verify_json_structure(task_text)
        if is_valid:
            break
        print(f"  ⚠️  第 {attempt+1} 次尝试失败：{task_data}")
        time.sleep(1)

    print(task_text)
    if is_valid:
        if isinstance(task_data, dict):
            task_data = [task_data]
        all_tasks.extend(task_data)
        print(f"✓ 生成成功，共 {len(task_data)} 个任务")
    else:
        print(f"✗ JSON 解析失败（已重试 {MAX_RETRIES} 次）: {task_data}")
    print('='*60)

# 保存
tasks_json_text = _json.dumps(all_tasks, ensure_ascii=False, indent=2)
tasks_path = GENERATED_DATA_DIR / "tasks.json"
save_generated_file(str(tasks_path), tasks_json_text)
print(f"\n✓ 共生成 {len(all_tasks)} 个任务，已保存到: {tasks_path}")


正在生成 purpose: 查最新薪资结构
{
  "id": "hr_latest_salary_structure_emp005_l2",
  "difficulty": "L2",
  "description": {
    "purpose": "查最新薪资结构",
    "relevant_policies": null,
    "notes": "需要先根据员工查询其全部薪资结构，再识别最新一条薪资结构记录。"
  },
  "user_scenario": {
    "persona": "王玲（EMP005，普通员工）",
    "instructions": {
      "task_instructions": "我想查一下我现在最新生效的薪资结构，帮我找到最新那条并告诉我关键信息。",
      "reason_for_call": "我想确认当前执行的薪资结构和基本工资金额是否正确。",
      "known_info": "我叫王玲，是员工 EMP005。",
      "unknown_info": "我不知道最新薪资结构的具体编号、基本工资金额和生效日期。",
      "domain": "HR"
    }
  },
  "initial_state": null,
  "evaluation_criteria": {
    "actions": [
      {
        "action_id": "action_1",
        "name": "get_employee_salary_structures",
        "arguments": {
          "employee_ref": "EMP005"
        },
        "compare_args": ["employee_ref"]
      },
      {
        "action_id": "action_2",
        "name": "get_salary_structure",
        "arguments": {
          "salary_structure_ref": "SALSTR015"
        },
        "compa

In [ ]:
import json as _json, re as _re
from synthesis.task_gen import verify_tasks, ToolDAGParser

with open(GENERATED_DATA_DIR / "tasks.json", 'r', encoding='utf-8') as f:
    tasks = _json.load(f)
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    domain_db = _json.load(f)
with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()

dag = ToolDAGParser().parse(tools_code)
metrics = verify_tasks(tasks, dag, base_data)

# ---- ID 合法性检查 ----
all_valid_ids = set()
for table_data in {**base_data, **domain_db}.values():
    if isinstance(table_data, dict):
        all_valid_ids.update(table_data.keys())

id_errors = []
for task in tasks:
    task_id = task.get("id", "?")
    for action in task.get("evaluation_criteria", {}).get("actions", []):
        for arg_key, arg_val in action.get("arguments", {}).items():
            if arg_key.endswith("_ref") and isinstance(arg_val, str):
                if arg_val not in all_valid_ids:
                    id_errors.append(f"任务 {task_id} | 工具 {action['name']} | {arg_key}={arg_val} → 不存在于任何表")

print("=" * 60)
print("【ID 合法性检查】")
if not id_errors:
    print("✓ 所有 *_ref 参数均为合法 ID")
else:
    print(f"❌ 发现 {len(id_errors)} 处无效 ID：")
    for e in id_errors:
        print(f"  - {e}")

# ---- 结构覆盖度 ----
print("\n=== 结构覆盖度（Pattern 分布）===")
for pattern, count in sorted(metrics["pattern_coverage"].items()):
    print(f"  {pattern}: {count}")

# ---- WRITE 工具覆盖率 ----
wc = metrics["write_tool_coverage"]
uncovered_writes = metrics.get("uncovered_write_tools", [])
print(f"\nWRITE 工具覆盖率: {wc:.1%} {'✓' if wc >= 0.8 else '⚠️'}（目标 ≥ 80%）")
if uncovered_writes:
    print("  未覆盖的 WRITE 工具：")
    for t in uncovered_writes:
        print(f"    - {t}")

# ---- 员工均匀度 ----
eu = metrics["employee_uniformity"]
print(f"员工均匀度 (CV): {eu['cv']:.2f} {'✓' if eu['cv'] <= 0.3 else '⚠️'}（目标 ≤ 0.3）")

# ---- WRITE 任务占比 ----
wr = metrics["write_task_ratio"]
print(f"WRITE 任务占比: {wr:.1%} {'✓' if wr >= 0.6 else '⚠️'}（目标 ≥ 60%）")

# ---- 难度分布 ----
print("\n=== 难度分布 ===")
total = len(tasks)
for level in ["L1", "L2", "L3", "L4", "L5"]:
    count = metrics["difficulty_distribution"].get(level, 0)
    print(f"  {level}: {count:4d} ({count/total:.1%})")
print(f"  总计：{total} 个任务")
